[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/liquid_crystals/liquid_crystals.ipynb)

# Nematic Liquid Crystals

This notebook computes orientational restoring torques and active defect motion in a two-dimensional nematic liquid-crystal film. Rod-like molecules align along a shared axis without distinguishing either end, so around a half-charge defect their axes turn through half a revolution while the degree of order melts smoothly to zero at the core. Passive elasticity draws opposite defects together until they annihilate, whereas substrate-dragged active stress propels the comet-like defect and stirs a disordered film ([active-defect mechanism](https://arxiv.org/abs/2209.00333)).

Alignment is modelled as a symmetric traceless map `order` on intrinsic directions, invariant under reversing any molecular axis. Infinitesimal rotation in a turning plane acts on both output and input via `order.commutator(Turn) - order(Point.commutator(Turn))`, so pairing against the molecular field yields a linear form on bivector turns whose restoring couple performs virtual work `turn & couple`. Simplex gradients `sides / double_areas` evaluate the active force and fluid vorticity directly on the triangulated film, while edge rotor logarithms of `order(reference)` count half-turn defect windings without choosing a director sign.

In coordinate Landau–de Gennes tensor notation, alignment is represented by the $2\times 2$ matrix $Q_{ij} = S(n_i n_j - \tfrac12\delta_{ij})$, orientational torque is extracted with the alternating symbol as $\tau = 2\varepsilon_{ij} Q_{ik} H_{kj}$, and half-charge windings are tracked through $\operatorname{atan2}(Q_{xy}, Q_{xx})$ angle differences around grid loops. Active local-drag flow couples $u_i = -\zeta\,\partial_j Q_{ij}$ back into $\partial_t Q_{ij} = H_{ij} - u_k\partial_k Q_{ij} + \Omega_{ik}Q_{kj} - Q_{ik}\Omega_{kj}$ via the antisymmetric vorticity matrix $\Omega_{ij} = \tfrac12(\partial_i u_j - \partial_j u_i)$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterable, Iterator

import jax
import numpy as np

from numga import Algebra, stack
from numga.backend.jax import JaxContext
from numga.sparse import SparseExtensor, spdiag
from numga.topology import Topology
from examples.util import shapes
from examples.util.animation import save_figure, show_states
from examples.fluids.liquid_crystals import render

ga = Algebra("x+y+")
mv = JaxContext(ga).multivector
Scalar = ga.gatype.scalar()                                      # Scalar+
Point = ga.gatype.antivector()                                   # Antivector-
Plane = ga.gatype.vector()                                       # Vector+
Turn = ga.gatype.bivector()                                      # Bivector+
Couple = ga.gatype.antibivector()                                # Antibivector-
Order = ga.gatype((Point, Point))                                # Point- <- Point-
Slope = ga.gatype((Scalar, Turn))                                # Scalar+ <- Turn+
work = Couple & Turn                                             # [] Scalar+ <- (Couple-, Turn+)

## Molecular alignment

An aligned patch stretches a direction along the molecular axis and shortens its perpendicular by the same amount. Reversing the axis leaves that map unchanged, so a half-charge defect needs no branch cut across which a signed director jumps; the alignment map stays continuous everywhere and shrinks smoothly to zero at the core.

In [ ]:
half_width, n_side = 6.0, 49
separation, core_radius = 2.4, 0.45
axis = np.linspace(-half_width, half_width, n_side)
topology, coordinates = shapes.box([axis, axis])
vertices = mv.vector(coordinates).dual().field()                 # Point-[V]
centres = mv.vector([[-separation / 2 + 0.03, 0.07],
                     [separation / 2 + 0.03, 0.07]]).dual()      # [2] Point-

In [ ]:
def alignment(directors: Point, strength: Scalar) -> Order:
    """Unit molecular axes `directors` and scalar `strength` define the symmetric traceless alignment
    map (`Point- <- Point-`)."""
    return strength * (directors * (directors.dual_inverse() & Point) - Point / 2)


def defect_pair(vertices: Point, centres: Point, core_radius: float, bearing: Turn) -> Order:
    """Two opposite half-charge defects at `centres` (`[2] Point-`) set an alignment field
    (`(Point- <- Point-)[V]`) whose order vanishes smoothly within `core_radius` of each core."""
    positive_offset, negative_offset = vertices - centres[0], vertices - centres[1]  # Point-[V], Point-[V]
    # The relative radial turn winds once at one core and backwards at the other. Its quarter power turns a
    # molecular axis through half that angle; crossing the logarithm's cut only reverses the axis.
    relative_rotor = positive_offset.normalized().dual_inverse() * negative_offset.normalized().dual_inverse()
    rotor = (relative_rotor.log() / 4 - bearing / 2).exp()
    directors = rotor >> mv.x.dual()                             # Point-[V]
    radii = positive_offset.norm() * negative_offset.norm()      # Scalar+[V]
    strength = radii / ((positive_offset.scalar_norm_squared() + core_radius ** 2) *
                        (negative_offset.scalar_norm_squared() + core_radius ** 2)).square_root()
    return alignment(directors, strength)


initial = defect_pair(vertices, centres, core_radius, mv.xy * 0)  # (Point- <- Point-)[V]

In [ ]:
def windings(topology: Topology, order: Order, reference: Point) -> Turn:
    """Edge rotor turns of `order(reference)` sum around each triangle into its director winding
    (`[...] Turn+[F]`) in whole turns."""
    # A traceless alignment acting on a fixed direction turns through twice the molecular angle. Its edge
    # logarithms therefore count half-charge defects without choosing a sign for any molecular axis.
    ends = topology.incidence[1, 0] * order(reference)[..., None]  # [..., 2] Point-[E]
    tails, heads = ends[..., 0], ends[..., 1]                      # [...] Point-[E], Point-[E]
    turns = (heads.dual_inverse() * tails.dual_inverse()).normalized().log()
    return (topology.d[1] * turns) / (4 * np.pi)


def defects(topology: Topology, vertices: Point, order: Order, reference: Point) -> tuple[Point, Scalar]:
    """Linear interpolation across each winding triangle locates the alignment zeros (`[...] Point-`)
    and reads their charges (`[...] Scalar+`) in whole turns."""
    charges = windings(topology, order, reference) & mv.xy.dual()  # Scalar+[F]
    winding_faces = np.flatnonzero(charges.norm() > 0.25)
    tails, heads = topology.incidence[1, 0] * order(reference)     # Point-[E], Point-[E]
    # The triangle of alignment values holds the origin at the weights its opposite sides span with it;
    # the same weights on the triangle's corners place the zero.
    zero_spans = (topology.boundary[2, 1] * (tails & heads)).batch()[:, winding_faces]  # [3, ...] Scalar-
    corner_weights = zero_spans / zero_spans.sum(axis=0)           # [3, ...] Scalar+
    corners = (topology.incidence[2, 0] * vertices).batch()[:, winding_faces]  # [3, ...] Point-
    return (corners * corner_weights).sum(axis=0), charges.batch()[winding_faces]


initial_defects = defects(topology, vertices, initial, mv.x.dual())
save_figure(render.draw_setup(topology, vertices, initial, initial_defects), "liquid_crystals_setup");

Fixed grain stretches symmetrically along the molecular axes and shortens where the blue tint marks reduced order at the cores. Copper and blue dots mark interpolated zeros of the continuous alignment field for positive and negative half-turn windings.

## Free energy

Bulk energy favours unit order while elastic energy penalizes variation between neighbouring vertices, giving each defect core a finite width. The term ***simplex gradient*** here means each triangle corner's opposite edge vector divided by twice the signed triangle area, `sides / double_areas`, which maps vertex values linearly to a constant gradient across the triangle. Minus the free-energy slope per unit area defines the symmetric traceless molecular field, which drives changes in both alignment strength and molecular orientation.

In Landau–de Gennes tensor notation, the free energy is $F = \int \bigl[\tfrac12 a\,(\operatorname{tr}(Q^2) - \tfrac12)^2 + \tfrac12 K\,\partial_k Q_{ij}\,\partial_k Q_{ij}\bigr]\,dA$, with symmetric traceless molecular field $H_{ij} = -\delta F/\delta Q_{ij}$.

In [ ]:
def triangles(topology: Topology, vertices: Point) -> tuple[Scalar, Plane, Scalar]:
    """Triangle geometry on `vertices` (`Point-[V]`) gives the triangle areas (`Scalar+[F]`),
    simplex gradients (`[3] Plane+[F]`), and lumped dual-cell areas (`Scalar+[V]`)."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)  # [3] Point-[F]
    double_areas = sides[0] & sides[1]                            # Scalar-[F]
    gradients = sides / double_areas                              # [3] Plane+[F]
    areas = double_areas.norm() / 2                               # Scalar+[F]
    dual_areas = (topology.incidence[2, 0].adjugate() * (areas / 3)).sum(axis=0)  # Scalar+[V]
    return areas, gradients, dual_areas


def elasticity(topology: Topology, areas: Scalar, gradients: Plane) -> SparseExtensor:
    """Triangle areas and simplex gradients assemble into the symmetric Dirichlet stiffness
    (`Scalar+[V <- V]`) on vertex fields."""
    corners = topology.incidence[2, 0]                            # [3] Scalar+[F <- V]
    local_stiffness = areas * (gradients[:, None] & gradients[None, :].dual())  # [3, 3] Scalar+[F]
    return (corners[:, None].adjugate() * spdiag(local_stiffness) * corners[None, :]).sum(axis=0).sum(axis=0)


elastic, bulk = 0.6, 3.0
areas, gradients, dual_areas = triangles(topology, vertices)
stiffness = elasticity(topology, areas, gradients)

In [ ]:
def free_energy(stiffness: SparseExtensor, dual_areas: Scalar, elastic: float, bulk: float, order: Order) -> Scalar:
    """Bulk order preference and elastic distortion integrate over `dual_areas` (`Scalar+[V]`) into the
    Landau–de Gennes free energy (`[...] Scalar+`)."""
    # Trace each local alignment map at its vertex without contracting across spatial sites.
    squared_order = order(order).batch().trace().field()          # [...] Scalar+[V]
    well = bulk * (squared_order - 0.5) ** 2 / 2                  # [...] Scalar+[V]
    distortion = order(stiffness * order).batch().trace().field()  # [...] Scalar+[V]
    return (dual_areas * well + elastic * distortion / 2).sites.sum()


def molecular_field(stiffness: SparseExtensor, dual_areas: Scalar, elastic: float, bulk: float,
                    order: Order) -> Order:
    """Minus the free-energy slope per unit area defines the symmetric traceless molecular field
    (`[...] (Point- <- Point-)[V]`)."""
    squared_order = order(order).batch().trace().field()          # [...] Scalar+[V]
    return -2 * bulk * (squared_order - 0.5) * order - elastic * (stiffness * order) / dual_areas

## Orientational derivative under a bivector turn

Rotating the molecules turns both the directions carried by the alignment map and the directions it reads. Pairing that infinitesimal change with the molecular field yields a linear form on bivector turns whose negative equals the virtual work `turn & couple` of the restoring couple. Because pure rotation preserves eigenvalues, this derivative captures orientational relaxation while leaving the core's degree of order unchanged.

In [ ]:
def turn_slope(order: Order, molecular: Order) -> Slope:
    """Commuting `order` with `Turn` and pairing against `molecular` gives the orientational energy
    slope (`[...] (Scalar+ <- Turn+)[V]`)."""
    # Both the direction carried by the map and the direction it reads turn with the molecules.
    tangent = order.commutator(Turn) - order(Point.commutator(Turn))
    # The trace pairs the molecular field with the map's change; the molecular field is minus the slope.
    return -molecular(tangent).batch().trace(0, 1).field()

In [ ]:
probe_width, imposed_turn, turn_span, n_turn_frames = 0.8, 1.0, 0.25, 61
frame_duration_ms = 40
# Displace a smooth patch near one defect out of alignment with its neighbours.
offset = vertices - (centres[0] + mv.y.dual() * probe_width)      # Point-[V]
probe = mv.xy * (-offset.scalar_norm_squared() / (2 * probe_width ** 2)).exp()  # Turn+[V]
imprint = (-probe * imposed_turn / 2).exp()
strained_order = imprint >> initial(imprint << Point)             # (Point- <- Point-)[V]
molecular = molecular_field(stiffness, dual_areas, elastic, bulk, strained_order)
slope = turn_slope(strained_order, molecular)                     # (Scalar+ <- Turn+)[V]
couple = work.solve(-slope)                                       # Couple-[V]
# The couple's virtual work is minus the first change of free energy.
predicted_slope = -(dual_areas * (probe & couple)).sites.sum()    # [] Scalar+
angles = np.linspace(-turn_span, turn_span, n_turn_frames)

The close-up camera isolates the distorted patch above the blue defect as virtual turns sweep through it. Drawing the animation below takes a few seconds.

In [ ]:
def with_defects(topology: Topology, vertices: Point, states: Iterable[Order],
                 reference: Point) -> Iterator[tuple[Order, list[tuple[Point, Scalar]]]]:
    """Yields each batched alignment state (`[...] (Point- <- Point-)[V]`) paired with the defect
    positions and charges of every film in the batch."""
    for order in states:
        yield order, [defects(topology, vertices, film, reference) for film in order]


def virtual_turns() -> Iterator[Order]:
    """Yields the `n_turn_frames` virtually rotated alignment states (`[1] (Point- <- Point-)[V]`)."""
    for angle in angles:
        rotor = (-probe * angle / 2).exp()
        yield (rotor >> strained_order(rotor << Point))[None]


turn_records = show_states(with_defects(topology, vertices, virtual_turns(), mv.x.dual()),
                           render.animate, topology, vertices, render.turn_camera,
                           name="liquid_crystals_turn", duration_ms=frame_duration_ms, colors=render.n_colours)
turn_states = [order for order, _ in turn_records]

In [ ]:
turn_energies = stack([free_energy(stiffness, dual_areas, elastic, bulk, state[0]) for state in turn_states])
baseline = free_energy(stiffness, dual_areas, elastic, bulk, strained_order)
save_figure(render.draw_derivative(angles, turn_energies, baseline, predicted_slope), "liquid_crystals_derivative");

The blue curve traces the exact free-energy change across the virtual turn sweep, and the dashed copper line shows the tangent `probe & couple` integrated over the film from a single bivector first derivative at zero angle.

## Relaxation by the bivector derivative

Rotational friction maps the orientational energy slope to a bivector angular velocity whose rotor turns the alignment at every vertex while holding its order strength fixed. Because only orientation relaxes, the prescribed defect cores remain pinned in place. Drawing the relaxation animation takes a few seconds.

In [ ]:
def turn_relaxation(topology: Topology, stiffness: SparseExtensor, dual_areas: Scalar, elastic: float, bulk: float,
                    initial: Order, time_step: float, n_frames: int, n_substeps: int) -> Iterator[Order]:
    """Yields `n_frames` alignment states (`[...] (Point- <- Point-)[V]`) as rotational friction relaxes
    the molecular orientation at fixed order strength."""
    # Unit rotational friction is a positive form on bivectors. Solving it against the negative derivative
    # gives the angular velocity whose dissipated power is the couple's work.
    friction = -Turn.scalar_product(Turn)                         # [] Scalar+ <- (Turn+, Turn+)
    boundary = topology.reading(topology.exterior[0], 0)          # Scalar+[V_exterior <- V]

    def step(order: Order, _) -> tuple[Order, None]:
        molecular = molecular_field(stiffness, dual_areas, elastic, bulk, order)
        angular_velocity = friction.solve(-turn_slope(order, molecular))
        angular_velocity = angular_velocity - boundary.adjugate() * (boundary * angular_velocity)
        rotor = (-angular_velocity * time_step / 2).exp()
        return rotor >> order(rotor << Point), None

    @jax.jit
    def advance(order: Order) -> Order:
        return jax.lax.scan(step, order, None, length=n_substeps)[0]

    order = initial
    yield order
    for _ in range(n_frames - 1):
        order = advance(order)
        yield order

In [ ]:
rotation_step, n_relaxation_frames, n_relaxation_substeps = 0.0075, 61, 2
rotation = turn_relaxation(topology, stiffness, dual_areas, elastic, bulk, strained_order[None],
                           rotation_step, n_relaxation_frames, n_relaxation_substeps)
rotation_records = show_states(with_defects(topology, vertices, rotation, mv.x.dual()),
                               render.animate, topology, vertices, render.turn_camera,
                               name="liquid_crystals_torque", duration_ms=frame_duration_ms, colors=render.n_colours)
rotation_states = [order for order, _ in rotation_records]

In [ ]:
rotation_energies = stack([free_energy(stiffness, dual_areas, elastic, bulk, state) for state in rotation_states])
rotation_times = np.arange(n_relaxation_frames) * rotation_step * n_relaxation_substeps
save_figure(render.draw_release(rotation_times, rotation_energies[:, 0]), "liquid_crystals_torque_energy");

The grain straightens as the molecular axes rotate down the orientational energy slope while the blue patches of reduced order stay pinned at the cores, releasing the free energy plotted above.

## Active stress and defect motion

Active molecules push or pull along their alignment axis, so spatial variation in `order` produces a net force that balances against local substrate drag to drive flow. That compressible flow advects the alignment and corotates it at half the fluid vorticity without flow-alignment strain coupling. Because activity continuously injects mechanical work, an active film need not descend the passive free energy.

In coordinate nematodynamics notation with zero flow-alignment parameter, local substrate drag gives $u_i = -\zeta\,\partial_j Q_{ij}$ and the alignment evolves as $\partial_t Q_{ij} = H_{ij} - u_k\partial_k Q_{ij} + \Omega_{ik}Q_{kj} - Q_{ik}\Omega_{kj}$ with vorticity matrix $\Omega_{ij} = \tfrac12(\partial_i u_j - \partial_j u_i)$.

In [ ]:
def active_velocity(topology: Topology, areas: Scalar, gradients: Plane, dual_areas: Scalar,
                    activity: np.ndarray, order: Order) -> Point:
    """Active stress divergence balanced against unit substrate drag drives the interior film velocity
    (`[...] Point-[V]`)."""
    corners = topology.incidence[2, 0]                            # [3] Scalar+[F <- V]
    face_order = corners * order[..., None]                       # [..., 3] (Point- <- Point-)[F]
    # Isotropic substrate drag turns the stress divergence into a velocity along its force.
    divergence = face_order(gradients.dual()).sum(axis=-1)        # [...] Point-[F]
    velocity = -activity * (corners.adjugate() * (areas * divergence / 3)[..., None]).sum(axis=-1) / dual_areas
    boundary = topology.reading(topology.exterior[0], 0)          # Scalar+[V_exterior <- V]
    return velocity - boundary.adjugate() * (boundary * velocity)

In [ ]:
def rate(topology: Topology, areas: Scalar, gradients: Plane, dual_areas: Scalar, stiffness: SparseExtensor,
         elastic: float, bulk: float, activity: np.ndarray, order: Order) -> Order:
    """Molecular relaxation, active advection, and vorticity corotation sum into the interior alignment
    rate (`[...] (Point- <- Point-)[V]`)."""
    molecular = molecular_field(stiffness, dual_areas, elastic, bulk, order)
    velocity = active_velocity(topology, areas, gradients, dual_areas, activity, order)
    corners = topology.incidence[2, 0]                            # [3] Scalar+[F <- V]
    local_order = corners * order[..., None]                      # [..., 3] (Point- <- Point-)[F]
    local_velocity = corners * velocity[..., None]                # [..., 3] Point-[F]
    face_velocity = local_velocity.mean(axis=-1)                  # [...] Point-[F]
    advection = (local_order * (gradients & face_velocity[..., None])).sum(axis=-1)
    velocity_gradient = (local_velocity * (gradients & Point)).sum(axis=-1)
    # Half the curl is the plane and rate of the fluid's rotation.
    rotation_rate = Point.reverse().commutator(velocity_gradient(Point)).batch().contract(1, 2).field() / 2
    face_order = local_order.mean(axis=-1)
    corotation = face_order.commutator(rotation_rate) - face_order(Point.commutator(rotation_rate))
    transport = (corners.adjugate() * (areas * (corotation - advection) / 3)[..., None]).sum(axis=-1) / dual_areas
    total_rate = molecular + transport
    boundary = topology.reading(topology.exterior[0], 0)          # Scalar+[V_exterior <- V]
    return total_rate - boundary.adjugate() * (boundary * total_rate)

With the perimeter alignment held fixed and zero boundary velocity, the passive film and two films of opposite activity start from the same defect pair and evolve side by side.

In [ ]:
activities = np.array([0.0, 0.4, -0.4])
time_step, n_substeps, n_frames = 0.015, 2, 91
initial_films = initial[None].broadcast_to((len(activities),))    # [3] (Point- <- Point-)[V]

In [ ]:
def evolution(topology: Topology, areas: Scalar, gradients: Plane, dual_areas: Scalar, stiffness: SparseExtensor,
              elastic: float, bulk: float, activity: np.ndarray, initial: Order, time_step: float,
              n_frames: int, n_substeps: int) -> Iterator[Order]:
    """Yields `n_frames` alignment states (`[...] (Point- <- Point-)[V]`) advanced by explicit midpoint
    substeps."""
    def step(order: Order, _) -> tuple[Order, None]:
        midpoint_order = order + time_step / 2 * rate(topology, areas, gradients, dual_areas, stiffness,
                                                      elastic, bulk, activity, order)
        next_order = order + time_step * rate(topology, areas, gradients, dual_areas, stiffness,
                                              elastic, bulk, activity, midpoint_order)
        return next_order, None

    @jax.jit
    def advance(order: Order) -> Order:
        return jax.lax.scan(step, order, None, length=n_substeps)[0]

    order = initial
    yield order
    for _ in range(n_frames - 1):
        order = advance(order)
        yield order

Stepping and rendering all three films together takes about ten seconds.

In [ ]:
films = evolution(topology, areas, gradients, dual_areas, stiffness, elastic, bulk, activities,
                  initial_films, time_step, n_frames, n_substeps)
film_records = show_states(with_defects(topology, vertices, films, mv.x.dual()),
                           render.animate, topology, vertices, render.defect_camera,
                           name="liquid_crystals_defects", duration_ms=frame_duration_ms, colors=render.n_colours)
states = [order for order, _ in film_records]

From top to bottom, the three rows show the passive film, positive activity, and negative activity in one shared window. Elastic attraction pulls the passive pair together until the two cores annihilate, whereas opposite activities propel the copper half-charge defect in opposite directions and alter when or whether the pair meets.

In [ ]:
energies = stack([free_energy(stiffness, dual_areas, elastic, bulk, state) for state in states])
times = np.arange(n_frames) * time_step * n_substeps
save_figure(render.draw_energy(times, energies), "liquid_crystals_energy");

The blue curve of the passive film decreases monotonically as elastic distortion relaxes and the defect pair annihilates, while the copper and green curves reflect active work competing with relaxation under opposite signs of activity.

## Crowded active film

Superposing random alignment waves and saturating their amplitude produces a smooth disordered film populated with half-charge defects of both signs wherever the wave sum vanishes. Under active substrate stress, nearby opposite pairs annihilate while surviving defects cruise through the rearranging domains.

In [ ]:
def disordered_alignment(vertices: Point, wavevectors: Plane, phases: np.ndarray,
                         amplitudes: Order, softness: float) -> Order:
    """Superposed plane waves (`[...] Plane+`) saturate into a smooth disordered alignment field
    (`(Point- <- Point-)[V]`) with half-charge defects where the wave sum vanishes."""
    waves = (wavevectors & vertices[..., None]) + phases         # [...] Scalar+[V]
    order = (amplitudes * waves.cos()).sum(axis=-1)              # (Point- <- Point-)[V]
    # Saturation gives strongly aligned domains while leaving each zero a continuous, disordered core.
    strength_squared = 2 * order(order).batch().trace().field()  # Scalar+[V]
    return order / (strength_squared + softness ** 2).square_root()

In [ ]:
dense_half_width, n_dense_side = 8.0, 65
n_waves, wave_number, softness, seed = 32, 1.8, 0.2, 7
dense_activities = np.array([1.2])
n_dense_substeps = 2
rng = np.random.default_rng(seed)
dense_axis = np.linspace(-dense_half_width, dense_half_width, n_dense_side)
dense_topology, dense_coordinates = shapes.box([dense_axis, dense_axis])
dense_vertices = mv.vector(dense_coordinates).dual().field()              # Point-[V_dense]
dense_wavevectors = mv.vector(rng.normal(size=(n_waves, 2)) * wave_number)  # [n_waves] Plane+
dense_phases = rng.uniform(0, 2 * np.pi, n_waves)
dense_bearings = mv.xy * rng.uniform(0, np.pi, n_waves)                   # [n_waves] Turn+
dense_directors = (-dense_bearings / 2).exp() >> mv.x.dual()              # [n_waves] Point-
dense_amplitudes = alignment(dense_directors, 1 / np.sqrt(n_waves))       # [n_waves] Point- <- Point-
dense_initial = disordered_alignment(dense_vertices, dense_wavevectors, dense_phases, dense_amplitudes, softness)
dense_defects = defects(dense_topology, dense_vertices, dense_initial, mv.x.dual())
save_figure(render.draw_dense_setup(dense_topology, dense_vertices, dense_initial, dense_defects),
            "liquid_crystals_dense_setup");

With the perimeter alignment pinned, the same free energy and substrate drag govern the crowded film. Stepping and drawing the animation below takes about half a minute.

In [ ]:
dense_areas, dense_gradients, dense_dual_areas = triangles(dense_topology, dense_vertices)
dense_stiffness = elasticity(dense_topology, dense_areas, dense_gradients)
dense_initial_films = dense_initial[None].broadcast_to((len(dense_activities),))  # [1] (Point- <- Point-)[V_dense]

In [ ]:
dense_films = evolution(dense_topology, dense_areas, dense_gradients, dense_dual_areas, dense_stiffness,
                        elastic, bulk, dense_activities, dense_initial_films, time_step, n_frames, n_dense_substeps)
dense_records = show_states(with_defects(dense_topology, dense_vertices, dense_films, mv.x.dual()),
                            render.animate, dense_topology, dense_vertices, render.dense_camera,
                            name="liquid_crystals_dense", duration_ms=frame_duration_ms, colors=render.n_colours)
dense_states = [order for order, _ in dense_records]

Copper and blue markers track the alignment zeros of opposite half-turn windings as close pairs annihilate and active stress sweeps the remaining defects across the film.

In [ ]:
dense_energies = stack([free_energy(dense_stiffness, dense_dual_areas, elastic, bulk, state)
                        for state in dense_states])
dense_times = np.arange(n_frames) * time_step * n_dense_substeps
save_figure(render.draw_energy(dense_times, dense_energies), "liquid_crystals_dense_energy");

Rapid early annihilation lowers the free energy sharply before active driving sustains ongoing domain rearrangement.

In [ ]:
# checks
# The two initial defects carry opposite half-turn charges.
_, initial_charges = initial_defects
assert ((initial_charges[initial_charges.argsort()] - np.array([-0.5, 0.5])).abs() < 1e-6).all()
# The bivector first derivative matches the central slope of the virtual-turn sweep.
midpoint = n_turn_frames // 2
central_slope = (turn_energies[midpoint + 1] - turn_energies[midpoint - 1]) / (angles[midpoint + 1] - angles[midpoint - 1])
assert (central_slope - predicted_slope).abs() < 1e-3
# Rotation driven by the bivector derivative dissipates energy while holding order strength fixed.
assert (rotation_energies[1:, 0] <= rotation_energies[:-1, 0]).all()
first_rotation, last_rotation = rotation_states[0].batch(), rotation_states[-1].batch()
assert ((first_rotation(first_rotation).trace() - last_rotation(last_rotation).trace()).abs() < 1e-6).all()
# Passive evolution dissipates free energy, annihilates the pair, and preserves symmetric traceless alignment.
assert (energies[1:, 0] <= energies[:-1, 0]).all()
_, passive_final_charges = film_records[-1][1][0]
assert len(passive_final_charges) == 0
last_order = states[-1].batch()
assert (last_order.trace().abs() < 1e-6).all()
assert np.allclose((last_order - last_order.adjoint()).kernel, 0, atol=1e-6)
# The crowded active film retains finite, symmetric traceless alignment.
assert dense_energies.isfinite().all()
dense_last_order = dense_states[-1].batch()
assert (dense_last_order.trace().abs() < 1e-6).all()
assert np.allclose((dense_last_order - dense_last_order.adjoint()).kernel, 0, atol=1e-6)